# BPIC17 (Start/Complete Filtered) vs Simulated Log

This notebook evaluates a simulated log generated by the **start_complete** dual-lifecycle model.

Both logs are filtered to lifecycle transitions in `{start, complete}` before comparison.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if (cwd / "integration").exists():
    repo_root = cwd
elif (cwd.parent / "integration").exists():
    repo_root = cwd.parent
else:
    repo_root = cwd

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from integration.SimulationBenchmark import SimulationBenchmark

print("Repo root:", repo_root)

In [ ]:
# Configure paths
ORIGINAL_LOG_PATH = repo_root / "Dataset" / "BPI Challenge 2017.xes"
# Update this if you store this simulation under a different name
SIMULATED_LOG_PATH = repo_root / "integration" / "output" / "simulated_log.csv"

if not ORIGINAL_LOG_PATH.exists():
    raise FileNotFoundError(f"Original log not found: {ORIGINAL_LOG_PATH}")
if not SIMULATED_LOG_PATH.exists():
    raise FileNotFoundError(f"Simulated log not found: {SIMULATED_LOG_PATH}")

print("Original log:", ORIGINAL_LOG_PATH)
print("Simulated log:", SIMULATED_LOG_PATH)

In [ ]:
# Load logs
import pm4py

orig_df = pm4py.convert_to_dataframe(pm4py.read_xes(str(ORIGINAL_LOG_PATH)))
sim_df = pd.read_csv(SIMULATED_LOG_PATH)

for df in [orig_df, sim_df]:
    if "time:timestamp" in df.columns:
        df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], format="mixed")

# Enforce start/complete filtering on BOTH logs
for name, df in [("orig", orig_df), ("sim", sim_df)]:
    if "lifecycle:transition" not in df.columns:
        raise ValueError(f"{name} log has no lifecycle:transition column")

orig_sc = orig_df[orig_df["lifecycle:transition"].astype(str).str.lower().isin(["start", "complete"])].copy()
sim_sc = sim_df[sim_df["lifecycle:transition"].astype(str).str.lower().isin(["start", "complete"])].copy()

print("Original events (filtered):", len(orig_sc), "| cases:", orig_sc["case:concept:name"].nunique())
print("Simulated events (filtered):", len(sim_sc), "| cases:", sim_sc["case:concept:name"].nunique())

In [ ]:
benchmark = SimulationBenchmark(
    original_log=orig_sc,
    simulated_log=sim_sc,
    filter_lifecycle_complete=False,
)

results = benchmark.compute_all_metrics()
results["simple_metrics"]

In [ ]:
results["basic_stats"]

In [ ]:
results["events_per_case"]

In [ ]:
results["throughput_time"]

In [ ]:
# Activity distribution comparison (top-20)
activity_cmp = results["activity_distribution"].copy()
activity_top20 = activity_cmp.sort_values("Original Count", ascending=False).head(20)

x = np.arange(len(activity_top20))
width = 0.42

plt.figure(figsize=(16, 6))
plt.bar(x - width/2, activity_top20["Original Share (%)"], width=width, label="Original (start/complete)")
plt.bar(x + width/2, activity_top20["Simulated Share (%)"], width=width, label="Simulated")
plt.xticks(x, activity_top20["Activity"], rotation=70, ha="right")
plt.ylabel("Share (%)")
plt.title("Start/Complete Activity Distribution: Original vs Simulated")
plt.legend()
plt.tight_layout()
plt.show()

activity_top20[["Activity", "Original Share (%)", "Simulated Share (%)", "Share Difference (%)"]]

In [ ]:
results["variants_comparison"].head(20)

In [ ]:
results["dfg_comparison"].head(25)

In [ ]:
print("Start activities")
display(results["start_activities"].head(20))

print("End activities")
display(results["end_activities"].head(20))

In [ ]:
# Optional export
OUT_XLSX = repo_root / "integration" / "output" / "bpic17_start_complete_vs_simulation_benchmark.xlsx"
benchmark.export_results(str(OUT_XLSX))
print("Exported:", OUT_XLSX)